In [1]:

from src.api.APIs import getInfoToposMSE,getInfoToposMSEWithoutCTC
import pandas as pd
import json
import regex
from pathlib import Path
import folium
from folium import plugins
from pyproj import Transformer
import requests
import folium
from folium import FeatureGroup, LayerControl
from pyproj import Transformer


In [2]:
BASE_URL = "http://info.api.elcano.operaciones.adif/stationsmanager/stations/"
def get_session():
    """Devuelve una sesión de requests reutilizable."""
    session = requests.Session()
    session.headers.update({
        "Accept": "application/json",
        "Content-Type": "application/json",
    })
    return session
 
def listar_estaciones(params=None, timeout=10):
    """
    Obtiene la lista completa de estaciones.
 
    Args:
        params (dict, opcional): Parámetros de query string para filtrar/paginar.
                                 Ejemplo: {"page": 1, "page_size": 50}
        timeout (int): Segundos de espera máximos para la respuesta.
 
    Returns:
        list | dict: Datos de la respuesta JSON, o None si hay error.
    """
    session = get_session()
    try:
        response = session.get(BASE_URL, params=params, timeout=timeout)
        response.raise_for_status()
        data = response.json()
        puntosId = data["data"]["list"]
        df = pd.DataFrame(puntosId)
        rename_columns = {
            "code": "Código",
            "name": "Nombre",
            "avmdld": "AVMDLD",
            "merchandise":"Mercancías",
            "suburban": "Cercanías",
            "national": "Nacional",
            "commercialArea":"Área Comercial",
            "commercial":"Comercial",
            "commercialMovements":"Movimientos Comercial",
            "sivType":"SIV"}
        df.rename(columns=rename_columns, inplace=True)
        return df
    except requests.exceptions.HTTPError as e:
        print(f"[ERROR HTTP] {e.response.status_code}: {e.response.text}")
    except requests.exceptions.ConnectionError:
        print("[ERROR] No se pudo conectar con la API. Verifica la red o la URL.")
    except requests.exceptions.Timeout:
        print("[ERROR] La solicitud superó el tiempo de espera.")
    except requests.exceptions.RequestException as e:
        print(f"[ERROR] {e}")
    return None
 
 

In [3]:
topo = getInfoToposMSE()

In [4]:
topo

,Delegación,Catálogo,CTC,NombreCTC,Tecnólogo,Código,Nombre,Mnemónico,Mnemónico_comercial,Red,MandoLocal,ConfiguraciónManual,Breteles,Sectores
0,SUR,MAL1-20180913E,MAL,Málaga,DIMETRONIC,54503,Guadalhorce,LG,GDH,RC,False,False,False,False
1,SUR,MAL1-20180913E,MAL,Málaga,DIMETRONIC,54412,Los Prados,LG,LG,RC,False,False,False,False
2,SUR,MAL1-20180913E,MAL,Málaga,DIMETRONIC,54511,Benalmadena,BD,BD,RC,False,False,False,False
3,SUR,MAL1-20180913E,MAL,Málaga,DIMETRONIC,54516,Fuengirola,FE,FE,RC,False,False,False,False
4,SUR,MAL1-20180913E,MAL,Málaga,DIMETRONIC,54517,Málaga Centro Alameda,ML,MCA,RC,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1787,CENTRO,OUR1,ORE,Ourense,ELIOP,31312,Vedra-Rivadulla,VR,VR,RC,False,False,False,False
1788,CENTRO,OUR1,ORE,Ourense,ELIOP,23006,Portas,XT,XT,RC,False,False,False,False
1789,CENTRO,OUR1,ORE,Ourense,ELIOP,31205,A Gudiña,AG,AG,RC,False,False,False,False
1790,CENTRO,OUR1,ORE,Ourense,ELIOP,22100,Ourense,OE,OE,RC,True,False,True,True


In [5]:
sin_topo = getInfoToposMSEWithoutCTC()

In [6]:
sin_ctc = sin_topo[~sin_topo["CTC"].isna()].copy()

In [7]:
sin_ctc.rename(columns={"AGER":"MandoLocal"}, inplace=True)

In [8]:
sin_ctc.drop(columns=["Delegación","ConfianzaLlegada","ConfianzaSalida","ConfianzaSalidaOrigen","Confianza de seguimiento","mseObservation"],inplace=True)

In [9]:
topo_total = pd.concat([topo, sin_ctc],ignore_index=True)

In [10]:
topo_total['Breteles'] = topo_total['Breteles'].fillna(False)
topo_total['Sectores'] = topo_total['Sectores'].fillna(False)

In [11]:
topo_total[topo_total["Nombre"] == "GIRONA"]

,Delegación,Catálogo,CTC,NombreCTC,Tecnólogo,Código,Nombre,Mnemónico,Mnemónico_comercial,Red,MandoLocal,ConfiguraciónManual,Breteles,Sectores


In [12]:
comercial = listar_estaciones()

In [13]:
comercial = comercial[["Código","Comercial"]].copy()

In [14]:
fname = Path(r"data/coordenadas.json")

In [23]:
with open(fname, "r", encoding="utf-8") as f:
    data = json.load(f)

# Extrae los campos de cada feature
records = [
    {
        "Código": f["properties"]["cod_depend"],
        "utm_x":      f["properties"]["utm_x"],
        "utm_y":      f["properties"]["utm_y"],
    }
    for f in data["features"]
]
df = pd.DataFrame(records)
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)
df["Longitud"], df["Latitud"] = transformer.transform(df["utm_x"].values, df["utm_y"].values)

In [24]:
df.loc[len(df)] = ["51420","","","-6.256101","36.521917"]

In [25]:
df

,Código,utm_x,utm_y,Longitud,Latitud
0,B0602,360326.8344,4617523.5355,-4.678554,41.697243
1,B1505,370011.2076,4655450.9394,-4.570551,42.040363
2,C5441,391783.5195,4110693.549,-4.218397,37.136355
3,C5442,412893.8349,4121118.024,-3.981969,37.232514
4,46A06,725839.8936,4369065.4736,-0.375689,39.441556
...,...,...,...,...,...
3081,A5116,213087.9734,4047606.3119,-6.204523,36.530826
3082,32000,270872.4372,4537288.7995,-5.722403,40.954619
3083,35211,286746.0994,4420148.828,-5.494688,39.904606
3084,40009,207657.3134,4302397.4636,-6.367256,38.821861


In [26]:
df_topo = pd.merge(
    topo,
    df[["Código","Latitud","Longitud"]],
    on= "Código",
    how= "left"
)


In [27]:
df_topo[df_topo["Latitud"].isna()]

,Delegación,Catálogo,CTC,NombreCTC,Tecnólogo,Código,Nombre,Mnemónico,Mnemónico_comercial,Red,MandoLocal,ConfiguraciónManual,Breteles,Sectores,Latitud,Longitud
222,CENTRO,CHA,MAC,Chamartín,DIMETRONIC,70211,Puerta Centro,PW,PW,RC,False,False,False,False,NaN,NaN
907,CENTRO,AV_MSM1,MSE,AV NAFA/MASE: Madrid-Sevilla,THALES,A3710,Ciudad Real -AG174,CIU,CRAG,AV,False,False,False,False,NaN,NaN
1783,CENTRO,OUR1,ORE,Ourense,ELIOP,32017,Vilagarcia de Arousa Praia,VC,VDAP,RC,False,False,False,False,NaN,NaN


In [28]:
df_topo.dropna(subset=["Latitud"], inplace=True)

In [29]:
df_topo = pd.merge(
    df_topo,
    comercial,
    on= "Código",
    how= "left"
)

In [55]:
df_topo['Comercial'] = df_topo['Comercial'].fillna(False)

In [56]:
df_agrupado = df_topo.groupby('Código').agg({
    'Nombre': 'first',
    'Latitud': 'first',
    'Longitud': 'first',
    'Red': lambda x: ', '.join(sorted(set(x.dropna().astype(str)))),
    'MandoLocal': 'first',
    'Breteles': 'first',
    'Sectores': 'first',
    'CTC': 'first',
    'Comercial': 'first'
}).reset_index()

In [49]:
df_agrupado[df_agrupado["Código"] == "37100"]

,Código,Nombre,Latitud,Longitud,Red,MandoLocal,Breteles,Sectores,CTC,Comercial
934,37100,Algodor,39.914235,-3.862967,RC,False,False,False,MAN,False


In [31]:
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\mapa.html")

In [60]:
import pandas as pd
import folium
from folium import FeatureGroup, LayerControl

# Agrupar estaciones duplicadas por Código y combinar sus redes
df_agrupado = df_topo.groupby('Código').agg({
    'Nombre': 'first',
    'Latitud': 'first',
    'Longitud': 'first',
    'Red': lambda x: ', '.join(sorted(set(x.dropna().astype(str)))),
    'MandoLocal': 'first',
    'Breteles': 'first',
    'Sectores': 'first',
    'CTC': 'first',
    'Comercial': 'first'
}).reset_index()

# Rellenar NaN con False para Breteles y Sectores
df_agrupado['Breteles'] = df_agrupado['Breteles'].fillna(False)
df_agrupado['Sectores'] = df_agrupado['Sectores'].fillna(False)

print(f"Estaciones originales: {len(df_topo)}")
print(f"Estaciones agrupadas: {len(df_agrupado)}")

# Versión con tile provider alternativo (CartoDB o Stamen)
mapa = folium.Map(
    location=[40.0, -3.0],
    zoom_start=6,
    tiles='CartoDB positron'
)

# Definir colores según la red (con combinaciones)
def get_colors_by_red(red):
    if pd.isna(red) or red == '':
        return '#808080', '#808080'
    
    red = str(red).strip().upper()
    
    # Combinaciones especiales (orden importa)
    if 'AV' in red and 'RC' in red:
        return '#FF69B4', '#089414'  # Borde Rosa, Relleno Verde (AV + RC)
    elif ('AV' in red and 'MIXTO' in red) or ('AV' in red and 'AM' in red):
        return '#FF69B4', '#FF8C00'  # Borde Rosa, Relleno Naranja (AV + MIXTO/AM)
    elif 'AM' in red and 'RC' in red:
        return '#0066CC', '#089414'  # Borde Azul, Relleno Verde (AM + RC)
    
    # Redes individuales
    colors_single = {
        'RC': ('#000000', '#089414'),    # Verde para RC
        'AV': ('#000000', '#FF69B4'),    # Rosa para AV
        'AM': ('#000000', '#0066CC'),    # Azul para AM
        'MIXTO': ('#000000', '#FF8C00')  # Naranja para Mixto
    }
    
    return colors_single.get(red, ('#808080', '#808080'))

# Obtener lista única de CTCs
ctcs_unicos = df_agrupado['CTC'].dropna().unique()
print(f"\nCTCs encontrados: {len(ctcs_unicos)}")

# Crear un grupo de capas para cada CTC
for ctc in ctcs_unicos:
    # Crear un FeatureGroup para este CTC
    ctc_group = FeatureGroup(name=f'CTC: {ctc}', show=False)  # Inicialmente desactivado
    
    # Filtrar estaciones de este CTC
    df_ctc = df_agrupado[df_agrupado['CTC'] == ctc].copy()
    
    if len(df_ctc) == 0:
        continue
    
    # Añadir marcadores para cada estación del CTC
    for idx, row in df_ctc.iterrows():
        border_color, fill_color = get_colors_by_red(row['Red'])
        
        # Verificar características
        tiene_mando_local = str(row['MandoLocal']).strip().lower() == 'true'
        tiene_breteles = str(row['Breteles']).strip().lower() == 'true'
        tiene_sectores = str(row['Sectores']).strip().lower() == 'true'
        es_comercial =str(row['Comercial']).strip().lower() == 'true'
        
        # Determinar la forma (círculo o cuadrado)
        if es_comercial:
            # Círculo
            forma_html = f"""
                <div style="
                    position: absolute;
                    top: 0;
                    left: 0px;
                    width: 12px;
                    height: 12px;
                    background-color: {fill_color};
                    border: 3px solid {border_color};
                    border-radius: 50%;
                    opacity: 0.9;
                "></div>
            """
        else:
            # Cuadrado
            forma_html = f"""
                <div style="
                    position: absolute;
                    top: 0;
                    left: 0px;
                    width: 12px;
                    height: 12px;
                    background-color: {fill_color};
                    border: 3px solid {border_color};
                    border-radius: 0%;
                    opacity: 0.9;
                "></div>
            """
        
        # Construir los iconos adicionales
        iconos_adicionales = ""
        width_extra = 0
        
        if tiene_mando_local:
            iconos_adicionales += """
                <div style="
                    position: absolute;
                    top: 2px;
                    left: 16px;
                    font-size: 14px;
                    filter: brightness(0) saturate(100%) invert(16%) sepia(99%) saturate(7404%) hue-rotate(3deg) brightness(95%) contrast(118%);
                ">👤</div>
            """
            width_extra += 14
        
        if tiene_breteles:
            iconos_adicionales += f"""
                <div style="
                    position: absolute;
                    top: 2px;
                    left: {16 + width_extra}px;
                    font-size: 14px;
                    font-weight: bold;
                    color: #000000;
                ">✗</div>
            """
            width_extra += 14
        
        if tiene_sectores:
            iconos_adicionales += f"""
                <div style="
                    position: absolute;
                    top: 1px;
                    left: {16 + width_extra}px;
                    font-size: 11px;
                    font-weight: bold;
                    color: #0066CC;
                ">A/B</div>
            """
            width_extra += 20
        
        # HTML para crear el icono
        total_width = 12 + width_extra
        icon_html = f"""
        <div style="position: relative; width: {total_width}px; height: 22px;">
            <div style="
                position: absolute;
                bottom: 0;
                left: 5px;
                width: 2px;
                height: 12px;
                background-color: #000000;
            "></div>
            {forma_html}
            {iconos_adicionales}
        </div>
        """
        
        # Tooltip con indicadores
        tooltip_text = f"{row['Nombre']} ({row['Red']})"
        if tiene_mando_local:
            tooltip_text += " 👤"
        if tiene_breteles:
            tooltip_text += " ✗"
        if tiene_sectores:
            tooltip_text += " A/B"
        
        folium.Marker(
            location=[row['Latitud'], row['Longitud']],
            popup=folium.Popup(
                f"<b>{row['Nombre']}</b><br>Código: {row['Código']}<br>CTC: {row['CTC']}<br>Red: {row['Red']}<br>"
                f"Comercial: {'Sí' if es_comercial else 'No'}<br>"
                f"Mando Local: {row['MandoLocal']}<br>Breteles: {row['Breteles']}<br>Sectores: {row['Sectores']}", 
                max_width=250
            ),
            tooltip=tooltip_text,
            icon=folium.DivIcon(html=icon_html, icon_size=(total_width, 22), icon_anchor=(6, 22))
        ).add_to(ctc_group)
    
    # Añadir el grupo al mapa
    ctc_group.add_to(mapa)

# Añadir leyenda
legend_html = '''
<div style="position: fixed; 
            bottom: 30px; right: 10px; width: 200px; height: 340px; 
            background-color: white; z-index:9999; font-size:12px;
            border:2px solid grey; border-radius: 5px; padding: 10px;
            overflow-y: auto;">
<p style="margin: 0; font-weight: bold;">Red Ferroviaria</p>
<p style="margin: 5px 0;"><span style="color: #089414;">●</span> RC - Red Convencional</p>
<p style="margin: 5px 0;"><span style="color: #FF69B4;">●</span> AV - Alta Velocidad</p>
<p style="margin: 5px 0;"><span style="color: #0066CC;">●</span> AM - Ancho Métrico</p>
<p style="margin: 5px 0;"><span style="color: #FF8C00;">●</span> Mixto</p>
<hr style="margin: 5px 0;">
<p style="margin: 5px 0; font-size: 11px;">🔴🟢 AV+RC (rosa/verde)</p>
<p style="margin: 5px 0; font-size: 11px;">🔴🟠 AV+MIXTO (rosa/naranja)</p>
<p style="margin: 5px 0; font-size: 11px;">🔵🟢 AM+RC (azul/verde)</p>
<hr style="margin: 5px 0;">
<p style="margin: 5px 0; font-size: 11px;">● Comercial</p>
<p style="margin: 5px 0; font-size: 11px;">■ No Comercial</p>
<hr style="margin: 5px 0;">
<p style="margin: 5px 0; font-size: 11px;"><span style="filter: brightness(0) saturate(100%) invert(16%) sepia(99%) saturate(7404%) hue-rotate(3deg) brightness(95%) contrast(118%);">👤</span> Mando Local</p>
<p style="margin: 5px 0; font-size: 11px; color: #000000; font-weight: bold;">✗ Breteles</p>
<p style="margin: 5px 0; font-size: 11px; color: #0066CC; font-weight: bold;">A/B Sectores</p>
</div>
'''
mapa.get_root().html.add_child(folium.Element(legend_html))

# Añadir botones de control para activar/desactivar todos los CTCs
control_buttons_html = '''
<div style="position: fixed; 
            top: 30px; right: 150px; 
            background-color: white; z-index:9999; 
            border:2px solid grey; border-radius: 5px; padding: 10px;">
<button onclick="toggleAllLayers(true)" style="
    padding: 8px 15px; 
    margin: 5px; 
    background-color: #4CAF50; 
    color: white; 
    border: none; 
    border-radius: 4px; 
    cursor: pointer;
    font-size: 12px;
">Visualizar Todos</button>
<button onclick="toggleAllLayers(false)" style="
    padding: 8px 15px; 
    margin: 5px; 
    background-color: #f44336; 
    color: white; 
    border: none; 
    border-radius: 4px; 
    cursor: pointer;
    font-size: 12px;
">Desactivar Todos</button>
</div>

<script>
function toggleAllLayers(show) {
    // Obtener el control de capas
    var layerControl = document.querySelector('.leaflet-control-layers');
    if (!layerControl) return;
    
    // Obtener todos los checkboxes de overlay
    var checkboxes = layerControl.querySelectorAll('.leaflet-control-layers-overlays input[type="checkbox"]');
    
    checkboxes.forEach(function(checkbox) {
        if (checkbox.checked !== show) {
            checkbox.click();
        }
    });
}
</script>
'''
mapa.get_root().html.add_child(folium.Element(control_buttons_html))

# Añadir control de capas para seleccionar CTCs
LayerControl(collapsed=False).add_to(mapa)

# Guardar el mapa
mapa.save(fname)
print(f"Mapa guardado en: {fname}")
print(f"Panel de control añadido con {len(ctcs_unicos)} CTCs")
print("Botones de control añadidos: 'Activar Todos' y 'Desactivar Todos'")

Estaciones originales: 1790
Estaciones agrupadas: 1748

CTCs encontrados: 27
Mapa guardado en: c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\mapa.html
Panel de control añadido con 27 CTCs
Botones de control añadidos: 'Activar Todos' y 'Desactivar Todos'
